In [ ]:
# this one is working!!!
# creating 1st datframe with list of flats

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import math

# Path to your ChromeDriver
chrome_driver_path = 'C:\\Users\\necha\\OneDrive\\Documents\\chromedriver-win64\\chromedriver.exe'

# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

# Set up Selenium WebDriver with options
service = Service(chrome_driver_path)
driver = webdriver.Chrome(service=service, options=options)

# Function to handle the consent screen
def handle_consent():
    try:
        # Wait for the consent button to be clickable
        WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.cw-btn.cw-btn--lg.cw-btn--green"))
        )
        consent_button = WebDriverWait(driver, 20).until(
    EC.element_to_be_clickable((By.XPATH, "//button[@data-testid='cw-button-non-targeted-ad']"))
)

        # Force click using JavaScript
        driver.execute_script("arguments[0].click();", consent_button)
        print("Consent handled: Clicked on 'Souhlasím'")

    except Exception as e:
        print("Could not find or interact with the consent screen:", str(e))

# Function to scrape a single page using Selenium
def scrape_page(url):
    driver.get(url)

    # Wait for the page to load completely
    time.sleep(5)  # Adjust the time as needed depending on your connection speed

    # Get the page source after JavaScript has rendered the content
    rendered_page = driver.page_source
    soup = BeautifulSoup(rendered_page, 'html.parser')

    # Extract locations, sizes, prices, and links
    locations = soup.select('span.locality.ng-binding')
    sizes = soup.select('span.name.ng-binding')
    prices = soup.select('span.norm-price.ng-binding')
    links = soup.select('a.title')

    page_data = []

    # Loop through items and extract data
    for i in range(len(locations)):
        location = locations[i].get_text().strip()
        size = sizes[i].get_text().strip()
        price = prices[i].get_text().replace('\xa0', '').replace('CZK', '').strip()  # Clean up price formatting
        link = 'https://www.sreality.cz' + links[i]['href']  # Full link

        # Append the data to the list
        page_data.append([location, size, price, link])

    return page_data

# Function to get total number of pages
def get_total_pages(url):
    driver.get(url)

    # Wait for the page to load completely
    time.sleep(5)

    # Get the page source and parse it
    rendered_page = driver.page_source
    soup = BeautifulSoup(rendered_page, 'html.parser')

    # Find all occurrences of 'span.numero.ng-binding'
    total_listings_tags = soup.find_all('span', class_='numero ng-binding')

    if len(total_listings_tags) > 1:
        # Select the second occurrence (index 1)
        total_listings = int(total_listings_tags[1].get_text().replace('\xa0', '').strip())
        print(f"Total listings found: {total_listings}")

        # Calculate total pages (20 listings per page)
        listings_per_page = 20
        total_pages = math.ceil(total_listings / listings_per_page)

    else:
        print("Could not find the total number of listings. Defaulting to 50 pages.")
        total_pages = 50  # Default to a high number like 50

    return total_pages

# Main scraping function to iterate through multiple pages dynamically
def scrape_all_pages():
    base_url = "https://www.sreality.cz/en/search/for-sale/apartments/praha"

    # First, handle the consent screen
    driver.get(base_url)

    # Ensure the consent screen is handled before proceeding
    handle_consent()

    # Then, get the total number of pages
    total_pages = get_total_pages(base_url)
    print(f"Total pages to scrape: {total_pages}")

    all_data = []

    # Loop through each page and scrape data
    for page in range(1, total_pages + 1):
        url = f"{base_url}?page={page}"
        print(f"Scraping page {page} of {total_pages}...")
        page_data = scrape_page(url)
        all_data.extend(page_data)  # Append page data to the main list

        time.sleep(2)  # Be polite to the server and avoid getting blocked

    return all_data

# Scrape all pages automatically
data = scrape_all_pages()

# Close the Selenium browser session
driver.quit()

# Convert the data to a DataFrame
df = pd.DataFrame(data, columns=["Location", "Size", "Price (CZK)", "Link"])

# Show the DataFrame
print(df)

# Optionally save the DataFrame to a CSV file
df.to_csv('apartments_praha.csv', index=False)






















import time
import pandas as pd
import signal
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import threading  # Import threading to handle locks

# Path to your ChromeDriver
chrome_driver_path = 'C:\\Users\\necha\\OneDrive\\Documents\\chromedriver-win64\\chromedriver.exe'

# Set up Chrome options (visible mode)
def get_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    service = Service(chrome_driver_path)
    return webdriver.Chrome(service=service, options=options)

# Function to handle the consent screen
def handle_consent(driver):
    try:
        WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.cw-btn.cw-btn--lg.cw-btn--green"))
        )
        consent_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@data-testid='cw-button-non-targeted-ad']"))
        )
        driver.execute_script("arguments[0].click();", consent_button)
        print("Consent handled: Clicked on 'Souhlasím'")
    except Exception:
        pass  # Suppress errors related to consent screen (visible or not)

# Function to scrape parameters
def scrape_params(driver):
    try:
        params_section = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.params.clear"))
        )
        groups = params_section.find_elements(By.CSS_SELECTOR, "ul")

        param_dict = {}
        for group in groups:
            items = group.find_elements(By.CSS_SELECTOR, "li")
            for item in items:
                param_label = item.find_element(By.CSS_SELECTOR, "label.param-label").text.strip(':')
                param_value = item.find_element(By.CSS_SELECTOR, "strong.param-value").text.strip()
                param_dict[param_label] = param_value

        return param_dict
    except Exception:
        return {}  # Return empty dictionary if parameters are missing

# Function to scrape a single page using an existing driver
def scrape_single_page(driver, link):
    driver.get(link)

    # Handle consent screen
    handle_consent(driver)

    # Scrape parameters
    params = scrape_params(driver)

    # Add link to params so we can merge it later
    params['Link'] = link
    return params

# Function to log progress every 100 flats scraped
def log_progress(scraped_count):
    if scraped_count % 100 == 0:
        logging.info(f"{scraped_count} flats scraped.")  # Change to log every 100 flats

# Global variable to track browser windows and scraped count
drivers = []
scraped_count = 0
count_lock = threading.Lock()  # Thread-safe lock

# Graceful shutdown function to close all browser windows
def graceful_shutdown(signum, frame):
    print("\nGraceful shutdown initiated...")
    for driver in drivers:
        driver.quit()
    print("All browser windows closed.")
    if signum is not None:
        exit(0)  # Only exit if triggered by signal (e.g., Ctrl+C)

# Register the signal handler
signal.signal(signal.SIGINT, graceful_shutdown)

# Function to scrape flats using fixed drivers (browser windows)
def scrape_pages_with_fixed_drivers(df, num_workers=5):
    global drivers, scraped_count
    # Initialize the drivers (one for each worker)
    drivers = [get_driver() for _ in range(num_workers)]

    # Handle consent on each driver manually the first time they open
    for driver in drivers:
        link = df.iloc[0]['Link']  # Load the first link into each driver
        driver.get(link)
        handle_consent(driver)

    all_params = []

    # Function to assign flats to a specific driver
    def scrape_for_driver(driver, links):
        global scraped_count  # Ensure that scraped_count is correctly referenced as global
        local_params = []
        for link in links:
            try:
                params = scrape_single_page(driver, link)
                local_params.append(params)

                # Update scraped count in a thread-safe way
                with count_lock:
                    scraped_count += 1
                    log_progress(scraped_count)  # Log progress after each flat
            except Exception as e:
                logging.error(f"Error scraping {link}: {e}")
        return local_params

    # Split the DataFrame into chunks for each driver
    chunks = [df[i::num_workers] for i in range(num_workers)]

    # Use ThreadPoolExecutor to assign chunks to each driver
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(scrape_for_driver, drivers[i], chunks[i]['Link']) for i in range(num_workers)]

        # Collect results as each driver finishes its chunk
        for future in as_completed(futures):
            result = future.result()
            all_params.extend(result)

    # Quit drivers after scraping is complete
    graceful_shutdown(None, None)

    return all_params

# Main scraping logic
if __name__ == "__main__":
    # Enable logging to track progress
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

    # Load the full dataset (all flats)
    df = pd.read_csv('apartments_praha.csv')

    # Configure number of browser windows (based on your system capacity)
    num_workers = 4  # Adjust this number based on your system capabilities

    # Scrape pages with the specified number of browser windows (manual consent, visible mode)
    try:
        scraped_params = scrape_pages_with_fixed_drivers(df, num_workers=num_workers)

        # Convert scraped data to DataFrame
        params_df = pd.DataFrame(scraped_params)

        # Merge the original DataFrame with the scraped data on the 'Link' column
        merged_df = pd.merge(df, params_df, on='Link', how='left')

        # Save the updated DataFrame to a CSV file
        merged_df.to_csv('updated_apartments_praha.csv', index=False)

        logging.info("Scraping completed and data saved to 'updated_apartments_praha.csv'")
    except KeyboardInterrupt:
        graceful_shutdown(None, None)
